# From HPC outputs to netcdf files

Thsi script is meant to take all of the outputs from the HPC and transform them in a netcdf file.


In [ ]:
import dask.dataframe as dd
import xarray as xr
import pandas as pd
import numpy as np
import dask.array as da

First run `merging_csv.ipynb` to produce a big csv file of your choice. 

Then I perform a merge with the rest of the pixel on earth so that I create a regular grid (`merged`). This is then written to memory as a csv file (it usually takes 11 mins).

In [ ]:
big_power_file = "world_5km_output_sumpower_w1.csv"

In [4]:
# open huge file with world power data
sumpower_csv = pd.read_csv(big_power_file) 
# remove execution time
sumpower_csv = sumpower_csv.drop(columns=['execution_time'])
print(sumpower_csv.head())

# Identify the value columns excluding lat and long
value_cols = sumpower_csv.columns.difference(['latitude', 'longitude'])

# Calculate total rect power by summing across the period
total_power = sumpower_csv[value_cols].sum(axis=1)

   latitude  longitude      33.1      35.4      38.0      40.7      43.6  \
0    62.125   -164.075  0.000030  0.000035  0.000036  0.000035  0.000036   
1    62.125   -164.025  0.000029  0.000037  0.000041  0.000043  0.000045   
2    62.125   -163.975  0.000031  0.000040  0.000045  0.000046  0.000046   
3    62.125   -163.925  0.000023  0.000029  0.000029  0.000027  0.000025   
4    62.125   -163.875  0.000022  0.000028  0.000030  0.000030  0.000029   

       46.8      50.1      53.7  ...    4860.5    5209.4    5583.3    5984.0  \
0  0.000038  0.000039  0.000038  ...  0.000004  0.000002  0.000003  0.000005   
1  0.000049  0.000050  0.000048  ...  0.000003  0.000001  0.000001  0.000002   
2  0.000046  0.000045  0.000041  ...  0.000002  0.000001  0.000002  0.000003   
3  0.000023  0.000022  0.000020  ...  0.000002  0.000002  0.000002  0.000003   
4  0.000027  0.000025  0.000024  ...  0.000003  0.000003  0.000005  0.000008   

     6413.5    6873.8        7367.2        7896.0        8462.

In [ ]:
# NOTE: This cell is to be run only if you want to create a new CSV file with proportions (normalised power values)

# # Create the proportion df, percentage of total power
# proportions = sumpower_csv[value_cols].div(total_power, axis=0)*100
# # Add back the latitude and longitude columns
# poportion_lat_long = sumpower_csv[['latitude', 'longitude']].join(proportions)
# poportion_lat_long.head()

# # save the new dataframe to a new CSV file so I can load it into memory if the kernel crashes
# poportion_lat_long.to_csv("world_5km_output_sumpower_w1_proportions.csv", index=False)

## Merging to create a regular grid
At this point you do not need the above code anymore, you can start from next cell without loading the previous

In [ ]:
# import the grid of points that are equally spaced, using dask because the files are huge and the kernel might crash
grid = dd.read_csv("pixel_centers_global.csv") # regular grid of lat long points extracted from the EVI netcdf files

to_merge = dd.read_csv("world_5km_output_sumpower_w1.csv") # This file was obtained from HPC results, by merging the 300 results from the job array on HPC
# if you want to do the proportions, load the proportions file instead
# to_merge = dd.read_csv("world_5km_output_sumpower_w1_proportions.csv")

In [ ]:
# Perform the left merge so that I will have a file with all coordinates on Earth. 
merged_dd = dd.merge(grid, to_merge, on=["latitude", "longitude"], how="left")
merged = merged_dd.compute()

In [ ]:
# check merged dataframe is correct and remove the execution time column
merged = merged.drop(columns=['execution_time'])
print(merged)

        latitude  longitude      33.1      35.4      38.0      40.7      43.6  \
0        -61.175   -142.975       NaN       NaN       NaN       NaN       NaN   
1        -61.175   -137.325       NaN       NaN       NaN       NaN       NaN   
2        -61.175   -132.125       NaN       NaN       NaN       NaN       NaN   
3        -61.175   -126.125       NaN       NaN       NaN       NaN       NaN   
4        -61.175    -83.925       NaN       NaN       NaN       NaN       NaN   
...          ...        ...       ...       ...       ...       ...       ...   
188786    59.225     37.675  0.000046  0.000052  0.000051  0.000048  0.000045   
188787    59.225     42.875  0.000038  0.000049  0.000054  0.000054  0.000054   
188788    59.225     42.925  0.000055  0.000070  0.000075  0.000069  0.000059   
188789    59.225     43.125  0.000039  0.000052  0.000059  0.000060  0.000058   
188790    59.225     46.425  0.000038  0.000043  0.000046  0.000048  0.000053   

            46.8      50.1 

In [ ]:
# save to disk merged dataframe, this is in case the kernel crashes
merged.to_csv("world_5km_output_sumpower_w1_merged.csv", index=False) # could take around 10 mins

## Convert to netcdf file

In [ ]:
file_to_convert = 'world_5km_output_sumpower_w1_merged.csv'
variable_name = "rect_power"
# variable_name = "proportions"  # if using proportions file
output_NetCDF_path = "world_5km_output_sumpower_w1_merged_v0.nc"

In [3]:
# Step 1: First pass to collect all unique coordinates
lat_set = set()
lon_set = set()

# Read CSV in chunks to collect all latitudes and longitudes
for chunk in pd.read_csv(file_to_convert, chunksize=10**5):
    lat_set.update(np.round(chunk['latitude'].unique(), 6))
    lon_set.update(np.round(chunk['longitude'].unique(), 6))


In [4]:
# Convert to sorted arrays
lats = np.sort(list(lat_set))
lons = np.sort(list(lon_set))

# Collect all unique periods
periods = np.sort([float(col) for col in pd.read_csv(file_to_convert, nrows=1).columns[2:]])


In [ ]:
# compute array dimensions
num_lats, num_lons, num_periods = len(lats), len(lons), len(periods)

# allocate NumPy arrays for each period
data_per_period = [np.zeros((num_lats, num_lons)) for _ in range(num_periods)]

The next step might take very long (11-15 hours).

In [ ]:
# fill the Dask array
for chunk in pd.read_csv(file_to_convert, chunksize=10**5):
    # extract latitude and longitude
    lats_chunk = chunk['latitude'].values
    lons_chunk = chunk['longitude'].values

    # find indices in the spatial grid using vectorized operations
    lat_indices = np.argmin(np.abs(lats[:, np.newaxis] - lats_chunk), axis=0)
    lon_indices = np.argmin(np.abs(lons[:, np.newaxis] - lons_chunk), axis=0)

    # for each period column in the chunk
    for col in chunk.columns[2:]:
        period = float(col)
        period_index = np.searchsorted(periods, period)  # Find index in sorted periods
        values = chunk[col].values  # Values for this period

        # assign values to the allocated arrays
        for i in range(len(lat_indices)):
            data_per_period[period_index][lat_indices[i], lon_indices[i]] = values[i]
            
# this took about 700 mins (11 hours) for me

In [ ]:
# stack the preallocated arrays into a 3D Dask array
dask_array = da.from_array(np.stack(data_per_period), chunks=(10, 100, 100))
print(dask_array)

dask.array<array, shape=(82, 3600, 7200), dtype=float64, chunksize=(10, 100, 100), chunktype=numpy.ndarray>


In [8]:
# then creating dask_array
dims = ["period", "lat", "lon"]
coords = {
    "period": periods,
    "lat": lats,
    "lon": lons,
}

# Wrap Dask array in xarray
data_array = xr.DataArray(dask_array, dims=dims, coords=coords, name=variable_name)
print(data_array.period)


<xarray.DataArray 'period' (period: 82)> Size: 656B
array([  33.1,   35.4,   38. ,   40.7,   43.6,   46.8,   50.1,   53.7,   57.6,
         61.7,   66.1,   70.9,   75.9,   81.4,   87.2,   93.5,  100.2,  107.4,
        115.1,  123.4,  132.2,  141.7,  151.9,  162.8,  174.5,  187. ,  200.4,
        214.8,  230.2,  246.7,  264.5,  283.4,  303.8,  325.6,  349. ,  374. ,
        400.8,  429.6,  460.5,  493.5,  528.9,  566.9,  607.6,  651.2,  697.9,
        748. ,  801.7,  859.2,  920.9,  987. , 1057.8, 1133.8, 1215.1, 1302.3,
       1395.8, 1496. , 1603.4, 1718.5, 1841.8, 1974. , 2115.7, 2267.5, 2430.3,
       2604.7, 2791.6, 2992. , 3206.8, 3436.9, 3683.6, 3948. , 4231.3, 4535. ,
       4860.5, 5209.4, 5583.3, 5984. , 6413.5, 6873.8, 7367.2, 7896. , 8462.7,
       9070.1])
Coordinates:
  * period   (period) float64 656B 33.1 35.4 38.0 ... 8.463e+03 9.07e+03


In [9]:
# Save to NetCDF
data_array.to_netcdf(output_NetCDF_path)